In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score,precision_score,recall_score
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import LabelEncoder
from transformers import (AutoTokenizer,AutoModelForSequenceClassification,TrainingArguments,Trainer,DataCollatorWithPadding,EarlyStoppingCallback)
from peft import (LoraConfig,get_peft_model,TaskType)

In [ ]:
LORA_DATASET_FILE = "../Dane/ALL_DATA_20.04.2026.csv" # ./source/pkd.csv | ../Dane/ALL_DATA_20.04.2026.csv 
MODEL_NAME = "sdadas/polish-roberta-base-v2"
MAX_LEN = 128
label_encoder = LabelEncoder()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir = "./model/")

In [47]:
data = pd.read_csv(LORA_DATASET_FILE, sep=';')
data = data[['PKD_2007','description_PL']]
data = data.rename(columns={
        'PKD_2007': 'kod_pkd', 
        'description_PL': 'opis'
    })
data['kod_pkd'] = data['kod_pkd'].astype(str).str.strip()
data['opis'] = (
        data['opis']
            .str.lower()
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
        )
data = data[data['opis'].str.len() > 20]
data = data[['kod_pkd', 'opis']].dropna()
data = data.drop_duplicates()
data["labels"] = label_encoder.fit_transform(data["kod_pkd"])
data.to_csv("labels_encoder.csv",index_label="id")
NUM_LABELS = len(label_encoder.classes_)
print(NUM_LABELS)

262


In [43]:
df_train, df_test = train_test_split(data,test_size=0.1,stratify=data["labels"])

In [44]:
df_train.to_csv('../Dane/teach_set.csv')
df_test.to_csv('../Dane/test_set.csv')

In [31]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(df_train["labels"]),
    y=df_train["labels"]
)

class_weights = torch.tensor(class_weights, dtype=torch.float)


In [32]:
class WeightedTrainer(Trainer):

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):

        labels = inputs.pop("labels")

        outputs = model(**inputs)

        logits = outputs.logits

        loss_fct = nn.CrossEntropyLoss(
            weight=class_weights.to(model.device)
        )

        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss
    

df_train["text"] = df_train["opis"].astype(str)
df_test["text"] = df_test["opis"].astype(str)
train_dataset = Dataset.from_pandas(
    df_train[["text", "labels"]]
)

test_dataset = Dataset.from_pandas(
    df_test[["text", "labels"]]
)

In [33]:
lengths = data["opis"].apply(lambda x: len(tokenizer.tokenize(x)))
print(lengths.describe())

count    79289.000000
mean        26.940862
std         34.066612
min          3.000000
25%         12.000000
50%         17.000000
75%         24.000000
max       1669.000000
Name: opis, dtype: float64


In [34]:
print(lengths.quantile([0.90, 0.95, 0.99, 0.995, 0.999]))

0.900     52.000
0.950     84.000
0.990    165.000
0.995    178.000
0.999    268.424
Name: opis, dtype: float64


In [35]:
counts = data["labels"].value_counts()

print(counts.describe())
print(counts.quantile([0.1,0.25,0.5,0.75,0.9,0.95,0.99]))

count     262.000000
mean      302.629771
std       277.316458
min         2.000000
25%       137.250000
50%       228.500000
75%       404.500000
max      1962.000000
Name: count, dtype: float64
0.10      41.20
0.25     137.25
0.50     228.50
0.75     404.50
0.90     587.20
0.95     694.95
0.99    1365.17
Name: count, dtype: float64


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS, cache_dir = "./model/")
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,

    target_modules=["query", "value", "key"],
    modules_to_save=["classifier"],
    bias="none"
)
model.config.use_cache = False
model.gradient_checkpointing_enable()
model = get_peft_model(model, peft_config)

model.print_trainable_parameters()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: sdadas/polish-roberta-base-v2
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 1,676,806 || all params: 126,321,164 || trainable%: 1.3274


In [18]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LEN,
        padding=False
    )

train_dataset = train_dataset.map(
    tokenize,
    batched=True,
    remove_columns=["text"]
)

test_dataset = test_dataset.map(
    tokenize,
    batched=True,
    remove_columns=["text"]
)

train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

test_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)


Map:   0%|          | 0/71360 [00:00<?, ? examples/s]

Map:   0%|          | 0/7929 [00:00<?, ? examples/s]

In [18]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
        "precision_macro": precision_score(labels,preds,average="macro",zero_division=0),
        "recall_macro": recall_score(labels,preds,average="macro",zero_division=0)
    }

In [21]:
# Parametryzacja treningu 
training_args = TrainingArguments(
    output_dir="./results",

    learning_rate=1e-4,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=8,

    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="f1_macro",

    logging_steps=50,
    
    save_total_limit=2,
    greater_is_better=True,
    report_to="none",
    warmup_steps=0.1,

    fp16=torch.cuda.is_available()
)
# Wybór trenera 
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
    
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [12]:
trainer.train()
merged_model = model.merge_and_unload()
merged_model.save_pretrained("./final_model")
tokenizer.save_pretrained("./final_model")

c:\Badania\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted,Precision Macro,Recall Macro
1,2.693309,2.452870,0.447219,0.375514,0.426812,0.396807,0.420042
2,1.922624,1.834065,0.565897,0.517469,0.560886,0.525744,0.555651
3,1.867074,1.631389,0.607391,0.566678,0.605784,0.571537,0.600724
4,1.685418,1.549593,0.628326,0.588096,0.626827,0.582367,0.621038
5,1.605905,1.521652,0.634506,0.597169,0.634429,0.591889,0.624847


c:\Badania\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\Badania\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\Badania\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\Badania\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./final_model\\tokenizer_config.json', './final_model\\tokenizer.json')

In [33]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_PATH = "./final_model"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH
)

trainer = Trainer(
    model=model,
    data_collator=DataCollatorWithPadding(tokenizer)
)

# prediction output
pred_output = trainer.predict(test_dataset)

# logits -> klasy
y_pred = np.argmax(pred_output.predictions, axis=-1)

# prawdziwe etykiety
y_true = pred_output.label_ids

print(confusion_matrix(y_true, y_pred))
report = classification_report(y_true, y_pred,output_dict=True)
print(report)
df_report = pd.DataFrame(report).transpose()

df_report.to_csv("classification_report.csv", index=True)

print(y_pred)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1450.07it/s]


[[46  4  6 ...  0  0  0]
 [ 4 43  2 ...  0  0  0]
 [ 9  3 41 ...  0  0  1]
 ...
 [ 0  0  0 ...  4  0  0]
 [ 0  0  0 ...  0  4  0]
 [ 0  0  0 ...  0  0 21]]
{'0': {'precision': 0.6865671641791045, 'recall': 0.696969696969697, 'f1-score': 0.6917293233082706, 'support': 66.0}, '1': {'precision': 0.7962962962962963, 'recall': 0.86, 'f1-score': 0.8269230769230769, 'support': 50.0}, '2': {'precision': 0.7068965517241379, 'recall': 0.5857142857142857, 'f1-score': 0.640625, 'support': 70.0}, '3': {'precision': 0.8690476190476191, 'recall': 0.6517857142857143, 'f1-score': 0.7448979591836735, 'support': 112.0}, '4': {'precision': 0.9767441860465116, 'recall': 0.9333333333333333, 'f1-score': 0.9545454545454546, 'support': 45.0}, '5': {'precision': 0.9375, 'recall': 0.8823529411764706, 'f1-score': 0.9090909090909091, 'support': 51.0}, '6': {'precision': 0.6190476190476191, 'recall': 0.5777777777777777, 'f1-score': 0.5977011494252874, 'support': 45.0}, '7': {'precision': 0.8, 'recall': 0.8571428571

c:\badania\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\badania\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\badania\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [34]:
from sklearn.metrics import top_k_accuracy_score

probs = torch.softmax(
    torch.tensor(pred_output.predictions),
    dim=-1
).numpy()

top3 = top_k_accuracy_score(
    y_true,
    probs,
    k=3,
    labels=np.arange(NUM_LABELS)
)

top5 = top_k_accuracy_score(
    y_true,
    probs,
    k=5,
    labels=np.arange(NUM_LABELS)
)
top10 = top_k_accuracy_score(
    y_true,
    probs,
    k=10,
    labels=np.arange(NUM_LABELS)
)

print(top3, top5, top10)

0.8085508891411275 0.8518098120822298 0.8968344053474587


### 

In [36]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_PATH = "./final_model"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH
)

trainer = Trainer(
    model=model,
    data_collator=DataCollatorWithPadding(tokenizer)
)

text = "Branża: Meble | Produkty: meble na wymiar | Usługi: projektowanie mebli"

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True
)

inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)

y_pred = outputs.logits.argmax(dim=-1).cpu().numpy()

print(y_pred)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 3006.62it/s]


[242]


In [37]:
data.loc[data['labels']==y_pred[0]].head(1)

,kod_pkd,opis,labels
285,3109,"branża: meble | produkty: łóżka do sypialni, m...",242
